# Chandraghanta — attack traffic in real labelled flows

Model 3 of the nine. **Data:** UNSW-NB15 and CIC-IDS2017 (research use only; weights are not published). Attach the Kaggle datasets `mrwellsdavid/unsw-nb15` and `chethuhn/network-intrusion-dataset`.

**Method:** LightGBM over each flow's features, threshold chosen on a validation slice of training. **Baseline:** PCA reconstruction error fitted on benign flows only.

**The point of this notebook:** CIC-IDS2017 split **by day** (train Monday to Wednesday, test Thursday and Friday, so every test attack family is unseen) against the random split most published results use. Results: `docs/research/models/chandraghanta.md`.

In [ ]:
import os
import subprocess
import sys

# Run on Kaggle or Colab. Clones this repository and installs the two packages.
if not os.path.exists("PASHUPATASTRA"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/preetraval45/PASHUPATASTRA.git"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "PASHUPATASTRA/packages/core",
                "-e", "PASHUPATASTRA/packages/sati", "matplotlib"], check=True)
sys.path[:0] = ["PASHUPATASTRA/packages/core", "PASHUPATASTRA/packages/sati"]

In [ ]:
from pathlib import Path

from sati.sources import netflow

flows = netflow.cic(Path("/kaggle/input"))
print(f"{len(flows):,} flows")
flows.groupby(["day", "family"]).size().unstack(fill_value=0).T

In [ ]:
import json
import subprocess
import sys

import pandas as pd

subprocess.run([sys.executable, "-m", "sati.netdetect.run", "--unsw", "/kaggle/input", "--cic", "/kaggle/input", "--out", "results/chandraghanta"], check=True, cwd="PASHUPATASTRA/packages/sati")
r = json.load(open("PASHUPATASTRA/packages/sati/results/chandraghanta/results.json"))
pd.DataFrame(r["datasets"]["cic"]["detectors"]).T

In [ ]:
import matplotlib.pyplot as plt

cic = r["datasets"]["cic"]
f1 = {"random split": cic["reference_random_split"]["chandraghanta-v0"]["f1"],
      "unseen days": cic["detectors"]["chandraghanta-v0"]["f1"],
      "PCA, unseen days": cic["detectors"]["pca (benign-only)"]["f1"]}
plt.bar(list(f1), list(f1.values()))
plt.ylabel("F1")
plt.title("The same model, two evaluations")
plt.show()

In [ ]:
import pandas as pd

families = pd.DataFrame(cic["per_family"]).T
families[families["attack"] == True]  # noqa: E712